In [1]:
%matplotlib inline
import random 
import torch
from d2l import torch as d2l 

# Generating the dataset 


In [2]:
# create y = Xw + b + e
# we assume e drawn from normal distribution with mean mu = 0 and standard deviation std = 1

class SyntheticRegressionData(d2l.DataModule): #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, 
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        
        # create synthetic data
        n = num_train + num_val
        self.X = torch.randn(n, len(w)) # this mean X have tensor size (n, len(w)) draw from normal distribution with mu = 0, std = 1
        noise = torch.randn(n, 1) * noise  # vector noise
        self.y = torch.matmul(self.X, w.reshape((-1,1))) + b + noise # reshape(-1,1) make sure that w is column vector

In [3]:
# test 
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

In [4]:
# let see the feature and the label 
print('feature:', data.X[0])
print('label', data.y[0])

feature: tensor([ 0.3021, -0.7970])
label tensor([7.5182])


# Reading the dataset


In [5]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    # get the indices
    if train: # create indies for training set 
        indices = list(range(0, self.num_train))
        # the examples are read in random order
        random.shuffle(indices) # need to shuffle
    else: # create indices for val set, does not need to shuffle
        indices = list(range(self.num_train, self.num_train + self.num_val))
    
    # get the batch 
    for i in range(0, len(indices), self.batch_size): 
        batch_indices = torch.tensor(indices[i: i + self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

In [9]:
X, y = next(iter(data.get_dataloader(train=True)))
print('X shape', X.shape)
print('y shape', y.shape)

X shape torch.Size([32, 2])
y shape torch.Size([32, 1])


# Concise Implementation of the Data 


In [ ]:
# using pytorch API to load data, it more efficiency 
# TensorDataset de dong goi du lieu 
@d2l.add_to_class(d2l.DataModule) #@save 
def get_tensorloader(self, tensors, train, indices=slice(0, None)): # slice = [:], None mean de trong la lay het
    
    # slice the data with  indices it get 
    # the tensors here is (self.X, self.y)
    tensors = tuple(a[indices] for a in tensors) # -> tensor(X matrix cat, y vector Cat)
    
    # dong goi thanh dataset
    dataset = torch.utils.data.TensorDataset(*tensors)  # split the  X matrix cat, y vector cat thanh tung thanh phan rieng biet 
    
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

# Exercise

1. Nếu số lượng mẫu không chia hết cho batch_size?

    Chuyện gì xảy ra: Batch cuối cùng sẽ có số lượng mẫu ít hơn các batch trước. Ví dụ: bạn có 100 mẫu, batch_size=32. Bạn sẽ có 3 batch đầu mỗi batch 32 mẫu, và batch cuối cùng chỉ có 100−(32×3)=4 mẫu.

    Cách thay đổi: Trong torch.utils.data.DataLoader, bạn có thể dùng đối số drop_last:

        drop_last=False (mặc định): Giữ lại batch cuối (dù nó nhỏ hơn).

        drop_last=True: Loại bỏ batch cuối cùng nếu nó không đủ kích thước. Điều này hữu ích khi một số thuật toán yêu cầu mọi batch phải có cùng kích thước (như Batch Normalization).


2. Khi dữ liệu quá lớn không thể chứa trong RAM?

    Hiện tượng: Nếu bạn cố nạp hết vào RAM, chương trình sẽ báo lỗi OutOfMemoryError hoặc máy tính bị treo (vì phải dùng bộ nhớ ảo Disk Swap rất chậm).

    Giải pháp:

        Sử dụng Memory Mapping (np.memmap trong NumPy): Truy cập file trên ổ cứng như thể nó đang ở trong RAM.

        Sử dụng Streaming: Đọc từng phần nhỏ của file từ ổ cứng khi cần (giống như xem video trực tuyến thay vì tải cả bộ phim về).


3. Shuffle dữ liệu khi nó nằm trên đĩa (Disk)?

    Vấn đề: Việc đọc ngẫu nhiên (random read) từng dòng trên ổ cứng rất chậm (đặc biệt là ổ HDD).

    Thuật toán hiệu quả:

        Sharded Shuffle: Chia dữ liệu thành nhiều file nhỏ (shards). Xáo trộn danh sách các file, sau đó nạp từng file vào RAM, xáo trộn trong RAM rồi mới đọc.

        Buffer Shuffle: Duy trì một bộ đệm (buffer) trong RAM. Đọc tuần tự từ đĩa vào đầy buffer, sau đó chọn ngẫu nhiên một mẫu từ buffer để đưa vào huấn luyện và thay thế nó bằng mẫu tiếp theo đọc tuần tự từ đĩa.